# CrewAI Debate: 찬반 토론 멀티 에이전트 프로젝트

이번 노트북에서는 CrewAI로 만든 실제 프로젝트인 **Debate(찬반 토론)** 샘플을 분석하고, 노트북에서 직접 실행해봅니다. 이전 노트북(03-1)에서 학습한 5단계 워크플로우가 실제로 어떻게 적용되었는지 확인합니다.

## 개요

| 주제 | 내용 |
|------|------|
| 프로젝트 구조 | CrewAI CLI로 생성된 표준 프로젝트 레이아웃 |
| agents.yaml | debater, judge 에이전트 설정 분석 |
| tasks.yaml | propose, oppose, decide 태스크 설정 분석 |
| crew.py | 데코레이터 패턴(@CrewBase, @agent, @task, @crew) 분석 |
| main.py | 실행 진입점 분석 |
| 직접 실행 | 노트북에서 동일한 Crew를 Python 코드로 구성하여 실행 |
| Custom Tool | BaseTool 상속 패턴 분석 |

## 학습 목표

1. CrewAI 프로젝트의 실제 구조를 파악하고 각 파일의 역할 이해하기
2. YAML 설정 파일에서 에이전트와 태스크가 어떻게 정의되는지 분석하기
3. 데코레이터 패턴으로 Crew가 어떻게 조립되는지 이해하기
4. Debate Crew를 직접 실행하고 결과를 확인하기
5. 같은 에이전트가 다른 태스크를 수행하는 패턴 이해하기

---

## 1. 프로젝트 구조

Debate 프로젝트는 `crewai create crew debate`로 생성된 표준 구조를 따릅니다.

```
debate/
├── pyproject.toml              # 프로젝트 설정 (crewai==1.6.1 의존성)
├── knowledge/
│   └── user_preference.txt     # Knowledge 소스 (RAG용)
├── output/                     # 태스크 결과물 저장
│   ├── propose.md              # 찬성 논거
│   ├── oppose.md               # 반대 논거
│   └── decide.md               # 최종 판결
└── src/debate/
    ├── config/
    │   ├── agents.yaml         # 에이전트 정의
    │   └── tasks.yaml          # 태스크 정의
    ├── tools/
    │   └── custom_tool.py      # 커스텀 도구 템플릿
    ├── crew.py                 # Crew 클래스 (핵심 파일)
    └── main.py                 # 실행 진입점
```

### Debate의 아이디어

주어진 **안건(motion)**에 대해:
1. **debater** 에이전트가 찬성 논거를 작성 (propose)
2. **같은 debater** 에이전트가 반대 논거를 작성 (oppose)
3. **judge** 에이전트가 양쪽 논거를 평가하고 판결 (decide)

---

## 2. agents.yaml 분석

에이전트의 역할, 목표, 배경 스토리를 정의합니다. `{motion}`은 실행 시 `inputs`로 전달되는 변수입니다.

```yaml
debater:
  role: >
    A compelling debater
  goal: >
    Present a clear argument either in favor of or against the motion.
    The motion is: {motion}
  backstory: >
    You're an experienced debator with a knack for giving concise
    but convincing arguments. The motion is: {motion}
  llm: openai/gpt-4o-mini

judge:
  role: >
    Decide the winner of the debate based on the arguments presented
  goal: >
    Given arguments for and against this motion: {motion},
    decide which side is more convincing, based purely on the
    arguments presented.
  backstory: >
    You are a fair judge with a reputation for weighing up arguments
    without factoring in your own views, and making a decision based
    purely on the merits of the argument. The motion is: {motion}
  llm: openai/gpt-4o-mini
```

**포인트**: debater는 하나의 에이전트이지만, 찬성(propose)과 반대(oppose) 두 태스크를 모두 수행합니다. 태스크의 description이 찬성/반대 역할을 구분해줍니다.

---

## 3. tasks.yaml 분석

각 태스크의 설명, 기대 출력, 담당 에이전트, 결과 파일을 정의합니다.

```yaml
propose:
  description: >
    You are proposing the motion: {motion}.
    Come up with a clear argument in favor of the motion.
    Be very convincing.
  expected_output: >
    Your clear argument in favor of the motion, in a concise manner.
  agent: debater
  output_file: output/propose.md

oppose:
  description: >
    You are in opposition to the motion: {motion}.
    Come up with a clear argument against the motion.
    Be very convincing.
  expected_output: >
    Your clear argument against the motion, in a concise manner.
  agent: debater
  output_file: output/oppose.md

decide:
  description: >
    Review the arguments presented by the debaters and
    decide which side is more convincing.
  expected_output: >
    Your decision on which side is more convincing, and why.
  agent: judge
  output_file: output/decide.md
```

**포인트**:
- `agent: debater` — YAML의 태스크 이름이 crew.py의 `@agent` 메서드 이름과 매칭됩니다
- `output_file` — 각 태스크 결과가 별도 파일로 저장됩니다
- sequential 프로세스이므로 oppose 태스크는 propose의 결과를 컨텍스트로 받고, decide는 둘 다 받습니다

---

## 4. crew.py 분석

데코레이터로 YAML 설정과 코드를 연결하는 핵심 파일입니다.

```python
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task


@CrewBase
class Debate():
    """Debate crew"""

    agents_config = 'config/agents.yaml'
    tasks_config = 'config/tasks.yaml'

    @agent
    def debater(self) -> Agent:
        return Agent(
            config=self.agents_config['debater'],
            verbose=True
        )

    @agent
    def judge(self) -> Agent:
        return Agent(
            config=self.agents_config['judge'],
            verbose=True
        )

    @task
    def propose(self) -> Task:
        return Task(
            config=self.tasks_config['propose'],
        )

    @task
    def oppose(self) -> Task:
        return Task(
            config=self.tasks_config['oppose'],
        )

    @task
    def decide(self) -> Task:
        return Task(
            config=self.tasks_config['decide'],
        )


    @crew
    def crew(self) -> Crew:
        """Creates the Debate crew"""
        return Crew(
            agents=self.agents,
            tasks=self.tasks,
            process=Process.sequential,
            verbose=True,
        )
```

### 코드 흐름 정리

```
@CrewBase
  └─ agents_config, tasks_config → YAML 자동 로드

@agent (debater, judge)
  └─ self.agents_config['debater'] → YAML 설정으로 Agent 생성
  └─ self.agents 리스트에 자동 추가

@task (propose, oppose, decide)
  └─ self.tasks_config['propose'] → YAML 설정으로 Task 생성
  └─ self.tasks 리스트에 자동 추가

@crew
  └─ self.agents + self.tasks → Crew 조립
```

---

## 5. main.py 분석

```python
from debate.crew import Debate

def run():
    inputs = {
        'motion': 'There needs to be strict laws to regulate LLMs',
    }
    result = Debate().crew().kickoff(inputs=inputs)
    print(result.raw)
```

실행 흐름: `Debate()` → `.crew()` → `.kickoff(inputs=inputs)`

- `inputs`의 `motion` 값이 YAML의 모든 `{motion}` 자리에 자동으로 삽입됩니다
- CLI에서는 `crewai run` 또는 `uv run debate`로 실행합니다

### 전체 실행 흐름

```
┌─────────────────────────────────────────────────────────────────┐
│              Debate 프로젝트 실행 흐름                          │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  inputs = {'motion': 'LLM 규제법이 필요하다'}                   │
│       │                                                         │
│       ▼                                                         │
│  ┌──────────────┐   찬성 논거 작성                              │
│  │  1. propose   │   Agent: debater                             │
│  │               │   Output → output/propose.md                 │
│  └──────┬───────┘                                               │
│         │ 컨텍스트 전달                                          │
│         ▼                                                        │
│  ┌──────────────┐   반대 논거 작성                               │
│  │  2. oppose    │   Agent: debater                              │
│  │               │   Output → output/oppose.md                   │
│  └──────┬───────┘                                                │
│         │ 컨텍스트 전달                                           │
│         ▼                                                         │
│  ┌──────────────┐   판결                                          │
│  │  3. decide    │   Agent: judge                                 │
│  │               │   Output → output/decide.md                    │
│  └──────────────┘                                                 │
│         │                                                         │
│         ▼                                                         │
│    CrewOutput (최종 판결 결과)                                     │
│                                                                   │
└───────────────────────────────────────────────────────────────────┘
```

---

## 6. Debate Crew 직접 실행

Debate 프로젝트의 핵심 로직을 노트북에서 직접 실행합니다. YAML 파일 없이 Python 코드로 동일한 Crew를 구성합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="pysbd")

from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found — .env 파일에 OPENAI_API_KEY를 설정하세요.")

In [ ]:
from crewai import Agent, Task, Crew, Process

# 안건 설정
motion = "There needs to be strict laws to regulate LLMs"

# Agent 정의 — agents.yaml과 동일한 설정을 Python 코드로
debater = Agent(
    role="A compelling debater",
    goal=f"Present a clear argument either in favor of or against the motion. The motion is: {motion}",
    backstory=f"You're an experienced debator with a knack for giving concise but convincing arguments. The motion is: {motion}",
    verbose=True,
    llm="openai/gpt-4o-mini",
)

judge = Agent(
    role="Decide the winner of the debate based on the arguments presented",
    goal=f"Given arguments for and against this motion: {motion}, decide which side is more convincing, based purely on the arguments presented.",
    backstory=f"You are a fair judge with a reputation for weighing up arguments without factoring in your own views, and making a decision based purely on the merits of the argument. The motion is: {motion}",
    verbose=True,
    llm="openai/gpt-4o-mini",
)

print(f"에이전트 생성 완료: {debater.role}, {judge.role}")

In [ ]:
# Task 정의 — tasks.yaml과 동일한 설정을 Python 코드로

propose_task = Task(
    description=f"You are proposing the motion: {motion}. Come up with a clear argument in favor of the motion. Be very convincing.",
    expected_output="Your clear argument in favor of the motion, in a concise manner.",
    agent=debater,
)

oppose_task = Task(
    description=f"You are in opposition to the motion: {motion}. Come up with a clear argument against the motion. Be very convincing.",
    expected_output="Your clear argument against the motion, in a concise manner.",
    agent=debater,
)

decide_task = Task(
    description="Review the arguments presented by the debaters and decide which side is more convincing.",
    expected_output="Your decision on which side is more convincing, and why.",
    agent=judge,
)

print("태스크 생성 완료: propose, oppose, decide")

In [ ]:
# Crew 구성 및 실행

debate_crew = Crew(
    agents=[debater, judge],
    tasks=[propose_task, oppose_task, decide_task],
    process=Process.sequential,
    verbose=True,
)

result = debate_crew.kickoff()

print("\n" + "="*60)
print("최종 판결:")
print("="*60)
print(result.raw)

In [ ]:
# 각 태스크의 결과를 개별적으로 확인

from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

console = Console()

for i, task_output in enumerate(result.tasks_output):
    titles = ["찬성 (Propose)", "반대 (Oppose)", "판결 (Decide)"]
    colors = ["green", "red", "blue"]
    console.print(Panel(
        Markdown(task_output.raw),
        title=titles[i],
        border_style=colors[i],
    ))
    console.print()

---

## 7. Custom Tool 분석

Debate 프로젝트에는 `tools/custom_tool.py`에 커스텀 도구 템플릿이 포함되어 있습니다. 이 프로젝트에서는 사용하지 않지만, 에이전트에게 외부 기능을 제공할 때 이 패턴을 따릅니다.

```python
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field


class MyCustomToolInput(BaseModel):
    """Input schema for MyCustomTool."""
    argument: str = Field(..., description="Description of the argument.")

class MyCustomTool(BaseTool):
    name: str = "Name of my tool"
    description: str = (
        "Clear description for what this tool is useful for, "
        "your agent will need this information to use it."
    )
    args_schema: Type[BaseModel] = MyCustomToolInput

    def _run(self, argument: str) -> str:
        # Implementation goes here
        return "this is an example of a tool output, ignore it and move along."
```

도구를 에이전트에 연결하려면 `@agent` 메서드에서 `tools` 파라미터를 추가합니다:

```python
@agent
def debater(self) -> Agent:
    return Agent(
        config=self.agents_config['debater'],
        tools=[MyCustomTool()],     # 도구 연결
        verbose=True
    )
```

---

## 정리

### Debate 프로젝트에서 배운 것

```
┌─────────────────────────────────────────────────────────────────────┐
│                Debate 프로젝트 핵심 요약                           │
├──────────────────────────────┬──────────────────────────────────────┤
│     파일                     │     역할                            │
├──────────────────────────────┼──────────────────────────────────────┤
│  agents.yaml                 │  에이전트의 role, goal, backstory   │
│  tasks.yaml                  │  태스크의 description, output       │
│  crew.py                     │  데코레이터로 YAML ↔ 코드 연결     │
│  main.py                     │  inputs 설정 + kickoff() 호출      │
│  custom_tool.py              │  BaseTool 상속 도구 템플릿          │
└──────────────────────────────┴──────────────────────────────────────┘
```

### 주요 패턴

- **같은 에이전트, 다른 태스크**: debater가 propose와 oppose를 모두 수행 — 태스크의 description이 역할을 구분
- **Sequential 컨텍스트 전달**: 이전 태스크 출력이 다음 태스크의 컨텍스트로 자동 전달
- **변수 치환**: YAML의 `{motion}`이 `inputs` 딕셔너리 값으로 자동 치환
- **output_file**: 각 태스크 결과를 별도 파일로 저장 가능

### CLI로 실행하기

```bash
cd agent_engineering/03_crew/debate
uv sync
crewai run
# 또는: uv run debate
```